In [1]:
import os
import shutil
from google.colab import drive

In [2]:
# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2. Install the official Ultralytics YOLO library
print(" Installing YOLOv11...")
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.4/107.7 GB disk)


In [ ]:
# --- [SURGICAL AI: YOLOv11 TRAINING PROTOCOL] ---

# 3. Transfer Data to Local SSD (Crucial for training speed)
print("\n Copying dataset from Google Drive to High-Speed Local SSD...")
drive_path = "/content/drive/MyDrive/DATASET_ICCSA_FINAL"
local_path = "/content/DATASET_ICCSA_FINAL"

if not os.path.exists(local_path):
    shutil.copytree(drive_path, local_path)
print(" Dataset successfully copied to local SSD!")

# 4. Fix YAML paths for the local SSD
yaml_path = f"{local_path}/data.yaml"
with open(yaml_path, 'r') as file:
    yaml_data = file.read()

# Update the path to point to the local SSD instead of Drive
yaml_data = yaml_data.replace(f"path: {drive_path}", f"path: {local_path}")

with open(yaml_path, 'w') as file:
    file.write(yaml_data)
print("YAML configuration updated for local training.")

# 5. START TRAINING!
# Using YOLOv11 small (yolo11s.pt) - balance of speed and precision
print("\n STARTING NEURAL NETWORK TRAINING...")
!yolo task=detect mode=train model=yolo11s.pt data={yaml_path} epochs=50 imgsz=640 batch=16 device=0 project=/content/drive/MyDrive/YOLO_RESULTS name=ICCSA_Run1

In [5]:
import pandas as pd
from ultralytics import YOLO

# 1. Unzip the dataset (-o overwrites, -q does it quietly and quickly)
print("[INFO] Unzipping the dataset. This will take a few seconds...")
!unzip -o -q "/content/drive/My Drive/project/DATASET_ICCSA_FINAL.zip" -d "/content/drive/My Drive/project/"

# 2. Automatically fix the path inside data.yaml
yaml_path = '/content/drive/My Drive/project/DATASET_ICCSA_FINAL/data.yaml'

print("[INFO] Updating paths in data.yaml...")
with open(yaml_path, 'r') as file:
    lines = file.readlines()

with open(yaml_path, 'w') as file:
    for line in lines:
        if line.startswith('path:'):
            # Force the correct absolute path to Drive
            file.write("path: /content/drive/My Drive/project/DATASET_ICCSA_FINAL\n")
        else:
            file.write(line)

# 3. Run validation and generate the table
MODEL_PATH = '/content/drive/My Drive/project/best.pt'
print("[INFO] Loading weights and running final validation...")
model = YOLO(MODEL_PATH)

# Run validation (without forced split to avoid folder errors)
metrics = model.val(data=yaml_path)

# Extract the data
class_indices = metrics.ap_class_index
precision = metrics.box.p
recall = metrics.box.r
map50 = metrics.box.map50
class_names = model.names

results_list = []
for i, class_id in enumerate(class_indices):
    name = class_names[class_id]
    results_list.append({
        "Class": name,
        "Precision (P)": f"{precision[i]:.3f}",
        "Recall (R)": f"{recall[i]:.3f}",
        "mAP@0.5": f"{map50[i]:.3f}"
    })

df_metrics = pd.DataFrame(results_list)

print("\n" + "="*60)
print(" PER-CLASS METRICS ")
print("="*60)
print(df_metrics.to_string(index=False))
print("="*60)

[INFO] Unzipping the dataset. This will take a few seconds...
[INFO] Updating paths in data.yaml...
[INFO] Loading weights and running final validation...
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11s summary (fused): 101 layers, 9,415,509 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.1 ms, read: 38.6±12.1 MB/s, size: 96.8 KB)
val: Scanning /content/drive/My Drive/project/DATASET_ICCSA_FINAL/labels/val... 878 images, 0 backgrounds, 8 corrupt: 100% ━━━━━━━━━━━━ 878/878 113.0it/s 7.8s
val: /content/drive/My Drive/project/DATASET_ICCSA_FINAL/images/val/robust_105553.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0222]
val: /content/drive/My Drive/project/DATASET_ICCSA_FINAL/images/val/robust_132087.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1493]
val: /content/drive/My Drive/project/DATASET_ICCSA_FINAL/images/val/robust_232212.jpg: 

IndexError: invalid index to scalar variable.